<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Project 2: Final Project

This project is inspired by the video ["Who's MJ statistical twin" by @Hooper_Stats](https://www.youtube.com/shorts/geYROqEcw8A). You will use the skills and concepts you've learned in the course to settle a classic sports-bar argument with data: **whose NBA career looks the most like Michael Jordan's?** You will use **linear regression**, **the bootstrap**, and **$k$-nearest neighbors** to find Jordan's "statistical twin" using the same 13 metrics as the video, and to ask what career statistics can (and can't) tell you about greatness, including who might be headed for the Hall of Fame.

---

## 🎗️ Assignment Reminders

As you work on this project, consider the following:
- **Initialize Otter.** 🚨 Make sure to run the code cell at the top of this notebook that starts with `# Initialize Otter` to load the auto-grader.
- **Complete the Tasks.** Your Tasks are categorized as auto-graded (📍) and manually graded (📍🔎).
    - For all the auto-graded tasks:
        - Replace the `...` in the provided code cell with your own code.
        - Run the `grader.check` code cell to run some tests on your code.
        - Keep in mind that for homework and project assignments, sometimes there are hidden tests that you will not be able to see the results of, which we use for scoring the correctness of your response. **Passing the auto-grader does not guarantee that your answer is correct.**
    - For all the manually graded tasks:
        - You might need to provide your own response to the provided prompt. Do so by replacing the template text "_Type your answer here, replacing this text._" with your own words.
        - You might need to produce a graphic or something else using code. Do so by replacing the `...` in the code cell to generate the image, table, etc.
        - In either case, [review the rubric](https://community.canvaslms.com/t5/Student-Guide/How-do-I-view-the-rubric-for-my-assignment/ta-p/275) on the associated <a href="https://ccsf.instructure.com" target="_blank">Canvas</a> Assignment page to understand the scoring criteria.
- **Code Sharing.** By submitting this project, you agree that you will not share your code directly with anybody but your partner. You are welcome to discuss questions with others but don't share the answers directly. The experience of solving the problems in this project will prepare you for our exams and potentially for future work in this field. If someone asks you for the answer, resist! Instead, you can demonstrate how you would solve a similar problem or you can focus on a specific part of a task.
- **Support.** You are not alone! Review the [Course Support page](https://ccsf-math-108.github.io/materials-fa26/resources/course-support.html) for how to get support in this course. If you're ever feeling overwhelmed or don't know how to make progress, talk to your instructor or a tutor.
- **Advice.** Develop your answers incrementally. To perform a complicated table manipulation, break it up into steps, perform each step on a different line, give a new name to each result, and check that each intermediate result is what you expect. You can add any additional names or functions you want to the provided cells. Make sure that you are using distinct and meaningful variable names throughout the notebook.
- **Variable Names.** Throughout this assignment and all future ones, please be sure to not re-assign variables throughout the notebook! _For example, if you use `max_temperature` in your answer to one question, do not reassign it later on. Otherwise, you will fail tests that you thought you were passing previously!_
- **Re-Submitting.** You may [submit](#🏁-Submit-Your-Assignment-to-Canvas) this assignment as many times as you want before the deadline. Your instructor will score the last version you submit once the deadline has passed.

---

## Set Up Notebook

### Task 00 📍

<!-- BEGIN QUESTION -->

Run the following code cells to set up the notebook.

_You are not responsible for understanding what this code does._

_Points:_ 0

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("project2.ipynb")

In [ ]:
# Configure this Notebook
import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import plotly.express as px
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

<!-- END QUESTION -->

---

## 📈 Section 1: Does Efficiency Predict Impact?

Michael Jordan played 15 seasons in the NBA for the Chicago Bulls and the Washington Wizards, winning six championships and five Most Valuable Player awards. Many fans call him the **GOAT**: the Greatest Of All Time. But "greatest" is hard to measure, so in this project you will focus on something you _can_ measure: the statistics each player put up over their career.

---

### The Data

The data in this project comes from [Basketball-Reference.com](https://www.basketball-reference.com), collected into CSV files by [Sumitro Datta](https://github.com/sumitrodatta/bball-reference-datasets). Every regular season of every player has been combined into one row per career. To keep comparisons fair, the table includes only:

* NBA regular seasons from 1979-80 (the first season with a 3-point line) through 2025-26, the same starting point as the video.
* Players whose first NBA season was 1979-80 or later, and who played at least 500 NBA games.

Run the following code cell to load the table. The rows are in a random order.

In [ ]:
careers = Table.read_table('careers.csv')
careers.show(5)

| Column | Description |
|---|---|
| `player` | The player's name |
| `first_season`, `last_season` | The first and last season played (a season is named by the year it ended, so 2003 means 2002-03) |
| `seasons`, `games` | The number of seasons and regular season games played |
| `ppg`, `apg` | Points and assists per game |
| `per` | **Player Efficiency Rating**: a summary of a player's per-minute box score production, scaled so that the league average is 15 |
| `bpm` | **Box Plus/Minus**: an estimate of how many points per 100 possessions a player adds compared with a league average player |
| `pts_100` ... `obpm` | The 13 metrics from the video, described in Section 2 |
| `hall_of_fame` | `Yes` if the player has been inducted into the Naismith Basketball Hall of Fame (for any reason), and `No` otherwise |
| `status` | `Retired` if the player's last season was 2019 or earlier, `Recent` if it was 2020-2025, and `Active` if the player played in 2025-26 |

---

### Task 1.1 📍🔎

<!-- BEGIN QUESTION -->

**Points per game** (`ppg`) is the most familiar basketball statistic: the total number of points a player scored in their career divided by the number of games they played. It measures how much a player scored, but not how efficiently or what else they contributed.

How does Jordan's scoring compare with everyone else's? Create a histogram of `ppg` for all careers using the bins `np.arange(0, 34, 2)`. The code provided then draws a vertical line at Jordan's career points per game.

_Points:_ 2

In [ ]:
...

# Keep this code to mark Jordan's career scoring average and give your image a title
jordan_ppg = careers.where('player', 'Michael Jordan').column('ppg').item(0)
plt.axvline(jordan_ppg, color='red', lw=3)
plt.title('Career Points per Game (Jordan in Red)')
plt.show()

<!-- END QUESTION -->

---

### PER and Box Plus/Minus

Points per game only tells you how much a player scored. Basketball analysts have also created statistics that try to sum up **everything** a player contributes in a single number. This section uses two of them. Both are built from a player's **box score**: the basic stats recorded in every game, such as points, rebounds, assists, steals, blocks, missed shots, and turnovers (losing the ball to the other team).

* **PER (Player Efficiency Rating)** adds up the good things a player does (scoring, rebounding, passing, steals, blocks) and subtracts the bad things (missed shots, turnovers, fouls), all per minute on the court. It is scaled so that an average NBA player has a PER of **15**. A PER above 20 is very good, and above 25 is an MVP-level season. Jordan's career PER was 27.9.

* **BPM (Box Plus/Minus)** also starts from the box score, but it estimates how much a player helps his team **outscore its opponents**. It is measured in points per 100 possessions compared with an average player. (A **possession** is one team's turn with the ball. Each team has roughly 100 possessions in a game, so "per 100 possessions" is about the same as "per game.") An average player has a BPM of **0**. A BPM of +5 means the player's team is estimated to do about 5 points better per game with him on the court than with an average player, and a negative BPM means the team does worse. Jordan's career BPM was 9.2.

Because both statistics try to measure the same thing, overall contribution, you would expect players with a high PER to also have a high BPM. In the rest of this section, you will use linear regression to describe how well PER predicts BPM, find out how far above the regression line Jordan sits, and use the bootstrap to estimate the slope of the true line.

---

### Task 1.2 📍🔎

<!-- BEGIN QUESTION -->

Create a scatter plot of `bpm` (vertical axis) against `per` (horizontal axis) for all careers in `careers`, including the regression line.

_Points:_ 2

In [ ]:
...

# Keep this code to give your image a title
plt.title('Box Plus/Minus vs. PER')
plt.show()

<!-- END QUESTION -->

---

### Task 1.3 📍

* Define a function `standard_units` that converts an array of numbers to standard units.
* Define a function `correlation` that takes a table `tbl` and two column labels, `x_label` and `y_label`, and returns the correlation coefficient $r$ between those two columns.
* Define functions `slope` and `intercept` that take the same arguments and return the slope and intercept of the regression line for predicting the `y_label` column from the `x_label` column.
* Assign `r_per_bpm`, `slope_per_bpm`, and `intercept_per_bpm` to the correlation, slope, and intercept for predicting `bpm` from `per` in `careers`.

**Note**: You can assume that the `x_label` and `y_label` columns contain only numbers, so you don't need to check for text or missing values.

_Points:_ 4

In [ ]:
def standard_units(arr):
    ...

def correlation(tbl, x_label, y_label):
    ...

def slope(tbl, x_label, y_label):
    ...

def intercept(tbl, x_label, y_label):
    ...

r_per_bpm = ...
slope_per_bpm = ...
intercept_per_bpm = ...

# Leave the following code to print out the calculated values.
print(f'r = {r_per_bpm:.4f}, and the regression line is  BPM = {slope_per_bpm:.4f} * PER + {intercept_per_bpm:.4f}')

In [ ]:
grader.check("task_1_3")

---

### Task 1.4 📍

Jordan's career PER was 27.9. Assign `jordan_predicted_bpm` to the BPM that the regression line predicts for Jordan, and `jordan_residual` to his actual BPM minus the predicted BPM.

_Points:_ 3

In [ ]:
jordan_row = careers.where('player', 'Michael Jordan')
jordan_predicted_bpm = ...
jordan_residual = ...

# Leave the following code to print out the calculated values.
print(f'Predicted BPM: {jordan_predicted_bpm:.2f}.  Residual: {jordan_residual:.2f}.')

In [ ]:
grader.check("task_1_4")

---

### Task 1.5 📍

Create a table called `bpm_fit` by adding a column called `'Residual'` to `careers` that contains each player's actual BPM minus the BPM predicted from their PER. Then assign `top_five_residuals` to a table of the 5 players with the **largest** residuals, showing only the columns `player`, `per`, `bpm`, and `Residual`.

_Points:_ 3

In [ ]:
bpm_fit = ...
top_five_residuals = ...
top_five_residuals

In [ ]:
grader.check("task_1_5")

---

### Task 1.6 📍🔎

<!-- BEGIN QUESTION -->

Create a residual plot: a scatter plot of `Residual` (vertical axis) against `per` (horizontal axis) using `bpm_fit`.

_Points:_ 2

In [ ]:
...

# Keep this code to give your image a title
plt.title('Residual Plot for Predicting BPM from PER')
plt.axhline(0, color='black', lw=1)
plt.show()

<!-- END QUESTION -->

The residuals form a band centered at 0 with no clear curve, so a straight line is a reasonable model. Now look back at `top_five_residuals`. Several of those players were known mainly for their **defense** or **passing**. BPM rates them higher than their PER predicts, which suggests that PER gives less credit for defense and playmaking than BPM does.

---

### Inference for the True Slope

If you think of these 830 careers as a sample of the players the NBA could have produced, the slope you found would change a bit with a different sample. In this part, you will estimate the slope of the true line with a bootstrap confidence interval.

---

### Task 1.7 📍

Define a function `bootstrap_slopes` that takes a table, two column labels, `x_label` and `y_label`, and a number of repetitions, and returns an array of the slopes from that many bootstrap resamples of the table.

Then use your function with 1,000 repetitions to assign `boot_slopes` to the array of bootstrap slopes, and assign `slope_ci_lower` and `slope_ci_upper` to the endpoints of an approximate 95% confidence interval for the true slope of the line for predicting `bpm` from `per`.

**Notes**
* You can assume that the `x_label` and `y_label` columns contain only numbers. 
* This code may take a few seconds to run.

_Points:_ 3

In [ ]:
def bootstrap_slopes(tbl, x_label, y_label, repetitions):
    ...

boot_slopes = ...
slope_ci_lower = ...
slope_ci_upper = ...

# Leave the following code to print out the calculated values.
print(f'Approximate 95% confidence interval for the true slope: [{slope_ci_lower:.4f}, {slope_ci_upper:.4f}]')

In [ ]:
grader.check("task_1_7")

---

### Task 1.8 📍

A friend says: "PER and BPM are basically unrelated, so the true slope is probably 0." Which is the best response, using a 5% cutoff? Assign `slope_conclusion` to the number (1, 2, or 3) of your choice.

1. Since 0 is not in the 95% confidence interval for the true slope, you reject the hypothesis that the true slope is 0. The data support a positive linear association between PER and BPM.
2. Since the confidence interval is narrow, you can't say anything about whether the true slope is 0.
3. Since a few players have large residuals, the regression line is meaningless, and you should accept that the true slope is 0.

_Points:_ 2

In [ ]:
slope_conclusion = ...
slope_conclusion

In [ ]:
grader.check("task_1_8")

---

## 👯 Section 2: Jordan's Statistical Twin

In Section 1, you saw that a single number like PER misses parts of a player's game, such as defense and passing. So how can you compare **whole careers**? In this section, you will find the career that is most similar to Jordan's across many statistics at once: his **statistical twin**.

---

### Measuring Distance

To find the career that is "closest" to Jordan's, you need a way to measure how far apart two careers are. If you describe each career by a few numbers, you can think of each career as a point and use the **Euclidean distance** between points, just like the distance formula you learned in algebra (but possibly with more than two coordinates).

You will start with just two numbers that every fan knows: points per game (`ppg`) and assists per game (`apg`). Run the following code cell to see every career, with Jordan in red. Hover over a point to see the player's name. Which players look closest to Jordan?

In [ ]:
jordan = careers.where('player', 'Michael Jordan')
fig = px.scatter(careers.to_df(), x='ppg', y='apg', hover_name='player', opacity=0.5,
                 title='Career Points and Assists per Game (Jordan in Red)')
fig.add_scatter(x=jordan.column('ppg'), y=jordan.column('apg'), mode='markers',
                marker=dict(color='red', size=12), hovertext=['Michael Jordan'], hoverinfo='text',
                name='Michael Jordan')
fig.show()

---

### Task 2.1 📍

Define a function `distance` that takes two arrays of the same length and returns the Euclidean distance between them.

_Points:_ 2

In [ ]:
def distance(arr1, arr2):
    ...

distance(make_array(1, 2), make_array(4, 6))

In [ ]:
grader.check("task_2_1")

---

### Task 2.2 📍

* Define a function `distances_to_jordan` that takes a table with a `player` column and an array of column labels. It should return an array with the distance between each row's values in those columns and Michael Jordan's values in those columns. (Jordan's own distance will be 0.)
* Use your function to assign `raw_closest` to a table of the 5 careers closest to Jordan's using `ppg` and `apg`. It should have the columns `player`, `ppg`, `apg`, and `'Distance'`, be sorted from smallest to largest distance, and **not** include Jordan.

**Hint**: `np.array(tbl.where('player', 'Michael Jordan').select(labels).row(0))` gives Jordan's values as an array. You can loop over `tbl.select(labels).rows` to get every row.

_Points:_ 3

In [ ]:
def distances_to_jordan(tbl, labels):
    ...

two_stats = make_array('ppg', 'apg')
raw_closest = ...
raw_closest

In [ ]:
grader.check("task_2_2")

---

### Standardizing

Look at the closest career in `raw_closest`. That player averaged far more assists than Jordan, so why are they closest? Points per game vary much more than assists per game (standard deviations of about 4.8 and 1.8), so a difference in points counts for much more in the distance than the same-sized difference in assists. In effect, `raw_closest` mostly measures who scored like Jordan.

To give each statistic an equal say, you can first convert each one to **standard units**, using the `standard_units` function you defined in Section 1.

---

### Task 2.3 📍

* Create a table called `two_su` with the column `player` from `careers`, followed by `ppg` and `apg` converted to standard units.
* Assign `su_closest` to a table of the 5 careers closest to Jordan's using the standardized `ppg` and `apg`, with the columns `player` and `'Distance'`, sorted from smallest to largest distance, not including Jordan.

_Points:_ 2

In [ ]:
two_su = ...
su_closest = ...
su_closest

In [ ]:
grader.check("task_2_3")

Standardizing changed the answer! Now a difference of one standard deviation in assists counts just as much as a difference of one standard deviation in points.

---

### Thirteen Metrics

Points and assists still tell only part of the story. The video that inspired this project used every season since 1979 and described each career with **13 metrics** that capture how a player scored, passed, rebounded, defended, and even how tall they were. Most of them are **rates**, measured per possession or per minute played, so players who played more minutes don't automatically look better.

| Column | Metric | What it measures |
|---|---|---|
| `pts_100` | Points per 100 possessions | Scoring volume |
| `ts_pct` | True shooting percentage | Scoring efficiency (counts 2s, 3s, and free throws) |
| `usg_pct` | Usage rate | Share of the team's possessions a player "uses" with a shot, free throws, or a turnover |
| `ft_rate` | Free-throw rate | Free throw attempts per field goal attempt |
| `three_rate` | 3-point attempt rate | Share of field goal attempts that are 3-pointers |
| `ast_pct` | Assist rate | Share of teammates' baskets a player assisted |
| `tov_pct` | Turnover rate | Turnovers per 100 plays |
| `trb_pct` | Rebound rate | Share of available rebounds a player grabbed |
| `blk_pct` | Block rate | Share of opponents' 2-point attempts a player blocked |
| `height` | Height | In inches |
| `stl_pct` | Steal rate | Share of opponents' possessions that end in a steal by the player |
| `dbpm` | Defensive Box Plus/Minus | Defensive points per 100 possessions above an average player |
| `obpm` | Offensive Box Plus/Minus | Offensive points per 100 possessions above an average player |

The game has changed a lot since 1980 (for example, teams now shoot far more 3-pointers), so the rates are **era-adjusted**: each season's value was converted to a difference from that season's league average before the seasons were combined. For example, Jordan's `three_rate` of $-3.8$ means that about 3.8% fewer of his shots were 3-pointers than the league average in the seasons he played.

---

### Task 2.4 📍

The video shows where Jordan ranks among all careers on each metric as a **percentile**. For example, if Jordan's percentile for steal rate is 98, then Jordan's steal rate is greater than or equal to the steal rates of 98% of the careers in the table.

**Step 1.** Define a function `percentile_rank` that takes two arguments, an array of numbers and a single value, and returns the **percent** (a number from 0 to 100) of the elements in the array that are less than or equal to the value. For example, `percentile_rank(make_array(1, 2, 3, 4), 3)` should return `75.0`, because 3 of the 4 elements are less than or equal to 3.

**Step 2.** Use your function to create a table called `jordan_profile` with one row for each of the 13 metrics and two columns:

* `'Metric'`: the metric labels, in the same order as the array `metrics`.
* `'Percentile'`: Jordan's percentile rank for that metric, comparing his value with the values for **all** careers in `careers`.

**Reminder**: Earlier in this section, you ran a cell that created the table `jordan`, which contains only Jordan's row of `careers`. For a metric `m`, `careers.column(m)` gives every player's values, and `jordan.column(m).item(0)` gives Jordan's value.

The code provided then plots Jordan's profile as a bar chart.

_Points:_ 3

In [ ]:
metrics = make_array('pts_100', 'ts_pct', 'usg_pct', 'ft_rate', 'three_rate', 'ast_pct', 'tov_pct',
                     'trb_pct', 'blk_pct', 'height', 'stl_pct', 'dbpm', 'obpm')

# Step 1
def percentile_rank(arr, value):
    ...

# Step 2
jordan_profile = ...

# Keep this code to plot Jordan's profile
jordan_profile.barh('Metric')
plt.xlim(0, 100)
plt.title("Michael Jordan's Percentiles")
plt.show()

In [ ]:
grader.check("task_2_4")

Your profile should look a lot like the one in the video: Jordan is at or near the top in scoring, usage, steals, and both kinds of Box Plus/Minus, and near the bottom in turnover rate. (A low turnover rate is a good thing!)

---

### Task 2.5 📍

Create a table called `career_su` with the columns `player`, `status`, and `hall_of_fame` from `careers`, followed by the 13 columns in `metrics`, each converted to standard units.

_Points:_ 2

In [ ]:
career_su = ...
career_su.show(3)

In [ ]:
grader.check("task_2_5")

---

### Task 2.6 📍

* Create a table called `jordan_distances` with the columns `player`, `status`, and `hall_of_fame` from `career_su`, plus a column called `'Distance'` that contains the distance between each career and Jordan's career, using all 13 metrics. Do not include Jordan himself in the table.
* Assign `jordan_twins` to a table of the 10 rows of `jordan_distances` with the smallest distances, sorted from smallest to largest.

**Hint**: Your `distances_to_jordan` function works with any number of columns.

_Points:_ 4

In [ ]:
jordan_distances = ...
jordan_twins = ...
jordan_twins

In [ ]:
grader.check("task_2_6")

The closest career is Jordan's **statistical twin**. In the video, the twin was **LeBron James**, but in your data he is number 3 on the list, only about 0.2 units farther away than Kawhi Leonard. The video doesn't say exactly how its distances were computed, and several of the closest players are still active, so their career statistics change every season.

How similar are Jordan and his twin? Run the following code cell to add the twin's percentiles to the profile you made in Task 2.4.

In [ ]:
twin = jordan_twins.column('player').item(0)
twin_row = careers.where('player', twin)
twin_percentiles = make_array()
for m in metrics:
    twin_percentiles = np.append(twin_percentiles, percentile_rank(careers.column(m), twin_row.column(m).item(0)))
both_profiles = jordan_profile.relabeled('Percentile', 'Michael Jordan').with_column(twin, twin_percentiles)
both_profiles.barh('Metric')
plt.xlim(0, 100)
plt.title('Jordan and His Statistical Twin')
plt.show()

The two profiles match closely on most metrics: both players were elite scorers and defenders who rarely turned the ball over. The biggest differences show where even a "twin" is not identical.

---

**Keep in mind**: Every "statistical twin" list depends on choices made by whoever built it: which metrics to use, which seasons and players to include, and how to scale the metrics. Different reasonable choices can give different answers, so a list like this is a summary of the data, not a fact about who is "most like" Jordan.

---

## 🏘️ Section 3: Predicting the Hall of Fame with $k$-Nearest Neighbors

In Section 2, you found the careers closest to Jordan's. A **$k$-nearest neighbors classifier** uses exactly the same idea to make predictions: to guess whether a player is a Hall of Famer, find the $k$ closest careers among players whose answer you already know, and go with the majority. You will use the 13 standardized metrics in `career_su` as features.

### The Hall of Fame

The **Naismith Memorial Basketball Hall of Fame** in Springfield, Massachusetts (the city where James Naismith invented basketball in 1891) has honored the sport's greatest players, coaches, referees, and contributors since 1959. Being inducted is the highest honor a basketball career can receive, and it is how fans and historians separate the great careers from the merely very good ones.

Players can't vote themselves in, and there is no formula. A player becomes eligible only after being retired for several full seasons. Then committees of experts (former players, coaches, executives, and journalists) review the candidates, and a finalist needs about 75% of the honors committee's votes to be inducted. Only a small fraction of NBA players ever make it: in this data, 49 of the 606 retired players are Hall of Famers.

That makes the Hall of Fame a great test for data science. It is a judgment made by people, based on everything they know about a career: statistics, but also championships, awards, playoff moments, and reputation. **Can a classifier that sees only the 13 metrics reproduce the experts' decisions?**

A few things to keep in mind:

* You will train and test the classifier on **Retired** players, whose careers ended in 2019 or earlier, so they have had several chances to be elected.
* The Hall of Fame honors a player's **whole** basketball career, not just their NBA career. Some players were inducted partly for international, college, or coaching accomplishments, so a few Hall of Famers have modest NBA statistics.

---

### Training, Validation, and Test Sets

Because the rows of `careers` are in a random order, you can split the **Retired** players into three groups:

* The first 300 players are the **training set**: the classifier finds neighbors among these players.
* The next 150 players are the **validation set**: you will use them to choose $k$.
* The remaining 156 players are the **test set**: you will use them only once, at the very end, to measure how well the final classifier works.

Run the following code cell to create them.

In [ ]:
retired_su = career_su.where('status', 'Retired')
train_set = retired_su.take(np.arange(300))
validation_set = retired_su.take(np.arange(300, 450))
test_set = retired_su.take(np.arange(450, retired_su.num_rows))
print('Training:', train_set.num_rows, ' Validation:', validation_set.num_rows, ' Test:', test_set.num_rows)

---

### Task 3.1 📍

A "lazy" classifier could predict `'No'` for every player. Assign `lazy_accuracy` to the proportion of players in `test_set` that this lazy classifier would classify correctly.

_Points:_ 2

In [ ]:
lazy_accuracy = ...
lazy_accuracy

In [ ]:
grader.check("task_3_1")

That is a very high accuracy for a classifier that never identifies a single Hall of Famer! Because so few players make the Hall of Fame, a classifier is only useful if it does **better** than this lazy classifier.

---

### Task 3.2 📍

* Define a function `closest` that takes a training table, a row of feature values (for example, `validation_set.select(metrics).row(0)`), a number `k`, and an array of feature labels. It should return the `k` rows of the training table closest to the row, with a `'Distance'` column, sorted by distance.
* Define a function `majority` that takes a table of neighbors and returns the value (`'Yes'` or `'No'`) that appears most often in its `hall_of_fame` column.
* Define a function `classify` that takes the same arguments as `closest` and returns the predicted value of `hall_of_fame` for the row.

**Hint**: Computing the distances in `closest` works just like your `distances_to_jordan` function from Section 2: loop over `training.select(feature_labels).rows` and use your `distance` function. `np.array(row)` turns a row into an array.

_Points:_ 3

In [ ]:
def closest(training, row, k, feature_labels):
    ...

def majority(neighbors):
    ...

def classify(training, row, k, feature_labels):
    ...

classify(train_set, validation_set.select(metrics).row(0), 5, metrics)

In [ ]:
grader.check("task_3_2")

---

### Task 3.3 📍

Define a function `accuracy` that takes a training table, a table of players to classify, a number `k`, and an array of feature labels. It should classify every player in the second table and return the proportion classified correctly. Then assign `k1_accuracy` to the accuracy on `validation_set` with $k = 1$.

**Hint: Review the thirteen metrics listed in [Task 2.4](#Task-2.4-📍)**

_Points:_ 3

In [ ]:
def accuracy(training, testing, k, feature_labels):
    ...

k1_accuracy = ...
k1_accuracy

In [ ]:
grader.check("task_3_3")

---

### Choosing $k$

You should choose $k$ without looking at the test set, so that it stays unseen until the end. Instead, you will try several values of $k$, measure each one's accuracy on the **validation set**, and pick the best.

---

### Task 3.4 📍

Create a table called `k_results` with a column `'k'` containing the odd numbers from 1 to 15 and a column `'Accuracy'` containing the accuracy on `validation_set` (using `train_set` as the training table and all 13 metrics) for each value of `k`. Then assign `best_k` to the value of `k` with the highest validation accuracy.

**Note**: This code may take 10 or more seconds to run.

_Points:_ 3

In [ ]:
k_values = np.arange(1, 16, 2)
k_results = ...
best_k = ...
best_k

# Leave this code to plot the validation accuracy for each k
k_results.plot('k', 'Accuracy')
plt.title('Validation Accuracy')
plt.show()

In [ ]:
grader.check("task_3_4")

---

### Task 3.5 📍

Assign `final_accuracy` to the accuracy on `test_set` using `best_k`.

_Points:_ 2

In [ ]:
final_accuracy = ...
final_accuracy

In [ ]:
grader.check("task_3_5")

---

### Who's on Track?

For fun, run the following code cell to point your classifier at players whose careers are still going. Keep in mind that it was trained on **retired** players with complete careers.

In [ ]:
active_su = career_su.where('status', 'Active')
active_predictions = make_array()
for row in active_su.select(metrics).rows:
    active_predictions = np.append(active_predictions, classify(train_set, row, best_k, metrics))
active_su.select('player').with_column('Prediction', active_predictions).where('Prediction', 'Yes').show()

---

### 🤔 Reflection

Consider the following questions. Your responses are not graded here, and you don't need to provide your answer in this notebook. However, your instructor may ask you to answer a question like this on the Final Exam, so it's a good idea to think it through or jot down notes now.

1. In Section 3, you chose $k$ using a **validation set** and used the **test set** only once, at the end. What would go wrong if you had instead tried each value of $k$ on the test set and reported the best test accuracy? Also, why does it make sense to use only **odd** values of $k$ for this classifier?
2. In Section 2, the closest career to Jordan changed when you converted points and assists per game to standard units. Explain why. Then suppose height had been recorded in centimeters instead of inches. Would Jordan's 13-metric twin change if the distances used the original units? Would it change if the distances used standard units?
3. Suppose you added a 14th feature to the Hall of Fame classifier that has nothing to do with being a Hall of Famer, such as a player's jersey number (converted to standard units). Would you expect the classifier's accuracy to go up, go down, or stay about the same? How could you check, without using the test set?
4. Suppose a bootstrap 95% confidence interval for the slope of the true line predicting BPM from PER had been $[-0.35, 0.23]$. Using a 5% cutoff, what could you conclude about the slope of the true line? For each of the following, which type of chart would be most appropriate: (a) the distribution of career points per game, (b) the relationship between PER and BPM, (c) the number of retired Hall of Famers and non-Hall of Famers?

---

## 🎉 Wrapping Up

Great work making it through this project! You used **linear regression** to see how two all-in-one statistics are related (and how far above the line Jordan sits), found Jordan's statistical twin using distances between standardized, era-adjusted metrics, used **the bootstrap** to put confidence intervals on a slope and on a classifier's accuracy, and built a **$k$-nearest neighbors classifier** from scratch.

Along the way, you also saw that data science is full of choices: which statistics to use, how to scale them, which players to include, and how to measure success. Good data scientists make those choices thoughtfully and say clearly what they chose.

---

## Final Task 📍

<!-- BEGIN QUESTION -->

You're almost done! Run the cell below to check that you passed all of the non-hidden auto-graded tasks, and then follow the instructions below to submit your assignment to Canvas.

_Points:_ 0

In [ ]:
grader.check_all()

<!-- END QUESTION -->

---

## 🏁 Submit Your Assignment to Canvas

1. **Review the Rubric:** View the rubric on the Canvas assignment page to understand the scoring criteria.  
2. **Review Manually Graded Tasks:** Verify that you have responded to all manually graded tasks marked with 📍🔎.  
3. **Review Auto-Graded Tasks:** Inspect the output from the `grader.check_all()` command above. Ensure you have passed all available tests for the auto-graded tasks in this notebook marked with 📍.  
   - This command executes all auto-grader tests sequentially.  
   - Note that Homework and Project assignments include hidden auto-grader tests that are not visible to you. Passing all visible tests does **not** guarantee that your solution is fully correct.  
4. **Save Your Work:** In the notebook toolbar, go to `File → Save default` to save your work and create a checkpoint.  
5. **Download the Notebook:** In the notebook toolbar, go to `File → Download IPYNB` to download the notebook (`.ipynb`) file.  
6. **Upload to Canvas:** Upload the downloaded `.ipynb` file to the appropriate Canvas assignment page.

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>